<a href="https://colab.research.google.com/github/yafiqzacky9-tech/tubes-Inventory-Reorder-Prediction./blob/main/tubes-Inventory-Reorder-Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

import joblib

In [34]:
df = pd.read_csv("inventory_dataset.csv")

df.head()

,item_id,nama_barang,kategori,supplier,gudang,satuan,harga_beli,harga_jual,stok_saat_ini,stok_minimum,stok_maksimum,avg_penjualan_harian,stockout_days,lead_time_hari,transaksi_30hari,terjual_30hari,persen_stok,days_of_supply,perlu_reorder
0,ITM-00001,Barang_1,Otomotif,CV Anugrah Perdana,Gudang D,set,126361,166548,381,51,357,19.91,20.0,5,8.0,672,106.72,19.14,1
1,ITM-00002,Barang_2,Makanan & Minuman,PT Maju Jaya,Gudang C,liter,217986,297460,85,85,425,2.40,12.0,2,5.0,77,20.00,35.42,1
2,ITM-00003,Barang_3,Pakaian,CV Anugrah Perdana,Gudang A,roll,202286,295602,274,51,306,17.18,5.0,27,64.0,364,89.54,15.95,1
3,ITM-00004,Barang_4,Otomotif,CV Anugrah Perdana,Gudang D,set,363320,413973,226,22,132,9.86,5.0,13,2.0,364,171.21,22.92,0
4,ITM-00005,Barang_5,Furnitur,PT Maju Jaya,Gudang B,pack,497508,593443,459,35,245,16.63,10.0,17,73.0,391,187.35,27.60,1


In [35]:
print(df.isnull().sum())

item_id                  0
nama_barang              0
kategori                 0
supplier                 0
gudang                   0
satuan                   0
harga_beli               0
harga_jual               0
stok_saat_ini            0
stok_minimum             0
stok_maksimum            0
avg_penjualan_harian    85
stockout_days           65
lead_time_hari           0
transaksi_30hari        72
terjual_30hari           0
persen_stok              0
days_of_supply           0
perlu_reorder            0
dtype: int64


In [36]:
df.fillna(df.median(numeric_only=True), inplace=True)

In [37]:
print("Duplicate:", df.duplicated().sum())

df.drop_duplicates(inplace=True)

Duplicate: 15


In [38]:
categorical_cols = [
    'kategori',
    'supplier',
    'gudang',
    'satuan'
]

le = LabelEncoder()

for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

In [39]:
df = df.drop([
    'item_id',
    'nama_barang'
], axis=1)

In [40]:
df.hist(figsize=(15,10))
plt.show()

In [41]:
plt.figure(figsize=(12,8))
sns.heatmap(df.corr(), annot=True)
plt.show()

In [42]:
plt.figure(figsize=(12,5))
sns.boxplot(data=df)
plt.xticks(rotation=90)
plt.show()

In [43]:
X = df.drop("perlu_reorder", axis=1)
y = df["perlu_reorder"]

In [44]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [45]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42
)

In [46]:
dt = DecisionTreeClassifier(random_state=42)

dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

print("Accuracy DT:",
      accuracy_score(y_test, y_pred_dt))

Accuracy DT: 0.988


In [47]:
print(classification_report(
    y_test,
    y_pred_dt
))

              precision    recall  f1-score   support

           0       0.93      0.97      0.95        58
           1       1.00      0.99      0.99       442

    accuracy                           0.99       500
   macro avg       0.96      0.98      0.97       500
weighted avg       0.99      0.99      0.99       500



In [48]:
cm = confusion_matrix(
    y_test,
    y_pred_dt
)

sns.heatmap(cm,
            annot=True,
            fmt='d')
plt.show()

In [49]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("Accuracy RF:",
      accuracy_score(y_test, y_pred_rf))

Accuracy RF: 0.988


In [50]:
print(classification_report(
    y_test,
    y_pred_rf
))

              precision    recall  f1-score   support

           0       0.95      0.95      0.95        58
           1       0.99      0.99      0.99       442

    accuracy                           0.99       500
   macro avg       0.97      0.97      0.97       500
weighted avg       0.99      0.99      0.99       500



In [51]:
cm = confusion_matrix(
    y_test,
    y_pred_rf
)

sns.heatmap(cm,
            annot=True,
            fmt='d')
plt.show()

In [52]:
dt_acc = accuracy_score(
    y_test,
    y_pred_dt
)

rf_acc = accuracy_score(
    y_test,
    y_pred_rf
)

print("Decision Tree :", dt_acc)
print("Random Forest :", rf_acc)

Decision Tree : 0.988
Random Forest : 0.988


In [53]:
joblib.dump(
    rf,
    "model_inventory.pkl"
)

joblib.dump(
    scaler,
    "scaler.pkl"
)

['scaler.pkl']

In [57]:
import streamlit as st
import joblib
import numpy as np

model = joblib.load("model_inventory.pkl")

st.title("Inventory Reorder Prediction")

stok_saat_ini = st.number_input(
    "Stok Saat Ini"
)

stok_minimum = st.number_input(
    "Stok Minimum"
)

avg_penjualan_harian = st.number_input(
    "Rata-rata Penjualan Harian"
)

lead_time_hari = st.number_input(
    "Lead Time Hari"
)

if st.button("Prediksi"):

    data = np.array([
        [
            stok_saat_ini,
            stok_minimum,
            avg_penjualan_harian,
            lead_time_hari
        ]
    ])

    hasil = model.predict(data)

    if hasil[0] == 1:
        st.error("Perlu Reorder")
    else:
        st.success("Tidak Perlu Reorder")

2026-06-15 04:29:35.720 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-15 04:29:35.723 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-15 04:29:35.724 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-15 04:29:35.726 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-15 04:29:35.729 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-15 04:29:35.731 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-15 04:29:35.734 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-15 04:29:35.737 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar